In [ ]:
import spacy
import pandas as pd
import numpy as np
import re
import utils
import random
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer


In [ ]:
df = pd.read_csv('gstt_pet_data_2012-22_anon.csv')
print(df.shape)

df_to_drop = pd.read_csv('tnm_all_uncertainty.csv')
acc_to_drop = df_to_drop['metadata.Accession'].to_list()

df = df[~df['Accession'].isin(acc_to_drop)].reset_index()

print(df.shape)

In [ ]:
df2 = df[df['Report text'].str.contains(r"(?s)^.*?(?=~)")]
print(df2.shape)

In [ ]:
df['Report text'][21853]

In [ ]:
ds_sample = pd.DataFrame(df['Report text'])

ds = Dataset.from_pandas(ds_sample)
ds[51]

In [ ]:
model_checkpoint = "UFNLP/gatortron-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples["Report text"])

tokenized_datasets = ds.map(tokenize_function, batched=True, num_proc=4, remove_columns=["Report text"])

In [ ]:
block_size = 512

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)



In [ ]:
lm_datasets = lm_datasets.train_test_split(test_size=0.1)
lm_datasets

In [ ]:
output_dir = "da_gatortron_base_GSTTv2"

model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
training_args = TrainingArguments(
                                output_dir=output_dir,
                                num_train_epochs=5,
                                save_strategy="epoch",
                                save_total_limit=3,
                                evaluation_strategy="epoch",
                                logging_dir=f"{output_dir}/logs",
                                logging_strategy="epoch",
                                metric_for_best_model='loss',
                                load_best_model_at_end=True,
                                learning_rate=1e-5,
                                per_device_train_batch_size=8,
                                per_device_eval_batch_size=8,
                                weight_decay=0.0,
                                label_names=["labels"],
                                push_to_hub=False,
                                )

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
trainer.model.save_pretrained(f"{output_dir}/results")
tokenizer.save_pretrained(f"{output_dir}/results")